# Feature engineering and exploratory data analysis

This notebook prepares the retail sales dataset for analysis and builds simple business-focused features.

In [1]:
# Import the libraries we will use
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Make charts appear inside the notebook
%matplotlib inline

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Find the dataset in the current workspace
repo_root = Path.cwd()
search_roots = [
    repo_root,
    repo_root.parent,
    Path("/home/vinuri/Documents/InsightRetail"),
]

possible_paths = []
for root in search_roots:
    possible_paths.extend([
        root / "data" / "raw" / "Online_Retail.xlsx",
        root / "data" / "raw" / "Online Retail.xlsx",
        root / "data" / "online_retail.xlsx",
        root / "Online_Retail.xlsx",
    ])

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find the retail dataset. Check the data folder.")

print(f"Loading dataset from: {DATA_PATH}")

# Load the Excel file
retail_df = pd.read_excel(DATA_PATH)
print("Dataset loaded successfully")
print(retail_df.head())

FileNotFoundError: Could not find the retail dataset. Check the data folder.

In [ ]:
# Clean column names for easier use
retail_df.columns = [col.strip().lower().replace(" ", "_") for col in retail_df.columns]
retail_df.head()

In [ ]:
# Convert the invoice date into a proper datetime column
retail_df["invoicedate"] = pd.to_datetime(retail_df["invoicedate"], errors="coerce")

# Keep cancelled transactions separately using invoices that start with 'C'
retail_df["is_cancelled"] = retail_df["invoiceno"].astype(str).str.startswith("C", na=False)

cancelled_df = retail_df[retail_df["is_cancelled"]].copy()
sales_df = retail_df[~retail_df["is_cancelled"]].copy()

print(f"Original rows: {len(retail_df)}")
print(f"Non-cancelled rows: {len(sales_df)}")
print(f"Cancelled rows: {len(cancelled_df)}")

In [ ]:
# Create new features for analysis
sales_df["salesamount"] = sales_df["quantity"] * sales_df["unitprice"]
sales_df["date"] = sales_df["invoicedate"].dt.date
sales_df["dayofweek"] = sales_df["invoicedate"].dt.day_name()
sales_df["month"] = sales_df["invoicedate"].dt.month_name()
sales_df["hour"] = sales_df["invoicedate"].dt.hour
sales_df["isweekend"] = sales_df["dayofweek"].isin(["Saturday", "Sunday"])
sales_df["yearmonth"] = sales_df["invoicedate"].dt.strftime("%Y-%m")

# Keep only rows with valid invoice dates for later analysis
sales_df = sales_df.dropna(subset=["invoicedate"])

sales_df.head()

In [ ]:
# Calculate the main business summary metrics
# These metrics help us understand the overall sales performance

total_revenue = sales_df["salesamount"].sum()
number_of_orders = sales_df["invoiceno"].nunique()
average_order_value = sales_df.groupby("invoiceno")["salesamount"].sum().mean()
total_customers = sales_df["customerid"].nunique()
total_products_sold = sales_df["quantity"].sum()
cancellation_rate = len(cancelled_df) / len(retail_df)

summary = {
    "total_revenue": total_revenue,
    "number_of_unique_orders": number_of_orders,
    "average_order_value": average_order_value,
    "total_customers": total_customers,
    "total_products_sold": total_products_sold,
    "cancellation_rate": cancellation_rate,
}

summary

In [ ]:
# Print the business summary in a simple format
print("Business summary")
print("-" * 30)
print(f"Total revenue: {total_revenue:,.2f}")
print(f"Number of unique orders: {number_of_orders:,}")
print(f"Average order value: {average_order_value:,.2f}")
print(f"Total customers: {total_customers:,}")
print(f"Total products sold: {total_products_sold:,.0f}")
print(f"Cancellation rate: {cancellation_rate * 100:.2f}%")

print("\nBusiness insight: Revenue is driven by a relatively small number of recurring orders, so improving repeat purchases can increase revenue.")

In [ ]:
# Revenue by day
revenue_by_day = sales_df.groupby("date")["salesamount"].sum().sort_index()

revenue_by_day.plot(figsize=(12, 4), title="Revenue by day")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: Daily revenue shows the pace of sales over time and helps identify short-term spikes or drops.")

In [ ]:
# Revenue by month
revenue_by_month = sales_df.groupby("month")["salesamount"].sum().reindex(["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"])

revenue_by_month.plot(kind="bar", figsize=(10, 4), title="Revenue by month")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: Monthly revenue helps the business understand seasonality and plan inventory or promotions.")

In [ ]:
# Revenue by country
revenue_by_country = sales_df.groupby("country")["salesamount"].sum().sort_values(ascending=False).head(10)

revenue_by_country.plot(kind="bar", figsize=(10, 4), title="Revenue by country")
plt.xlabel("Country")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: The strongest markets should receive more attention, while weaker regions may need targeted promotion.")

In [ ]:
# Revenue by weekday
revenue_by_weekday = sales_df.groupby("dayofweek")["salesamount"].sum().reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

revenue_by_weekday.plot(kind="bar", figsize=(10, 4), title="Revenue by weekday")
plt.xlabel("Weekday")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: Weekday demand patterns can guide staffing, delivery schedules, and marketing campaigns.")

In [ ]:
# Revenue by hour
revenue_by_hour = sales_df.groupby("hour")["salesamount"].sum().sort_index()

revenue_by_hour.plot(kind="bar", figsize=(10, 4), title="Revenue by hour")
plt.xlabel("Hour")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: Peak sales hours can help the team improve support coverage and campaign timing.")

In [ ]:
# Top 10 products by revenue
product_revenue = sales_df.groupby("description")["salesamount"].sum().sort_values(ascending=False).head(10)

product_revenue.plot(kind="bar", figsize=(10, 4), title="Top 10 products by revenue")
plt.xlabel("Product")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: The most profitable products should be promoted more heavily and stocked carefully.")

In [ ]:
# Top 10 products by quantity
product_quantity = sales_df.groupby("description")["quantity"].sum().sort_values(ascending=False).head(10)

product_quantity.plot(kind="bar", figsize=(10, 4), title="Top 10 products by quantity")
plt.xlabel("Product")
plt.ylabel("Quantity")
plt.tight_layout()
plt.show()

print("Business insight: Products sold in large volume may need stronger supply planning and better inventory control.")

In [ ]:
# Top 10 customers by revenue
customer_revenue = sales_df.groupby("customerid")["salesamount"].sum().sort_values(ascending=False).head(10)

customer_revenue.plot(kind="bar", figsize=(10, 4), title="Top 10 customers by revenue")
plt.xlabel("Customer ID")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

print("Business insight: High-value customers are strong candidates for loyalty offers and personalized marketing.")

In [ ]:
# Save the cleaned dataset for future modeling work
output_dir = repo_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Keep the main cleaned dataset for non-cancelled sales analysis
cleaned_output_path = output_dir / "cleaned_retail_data.csv"
sales_df.to_csv(cleaned_output_path, index=False)

print(f"Cleaned dataset saved to: {cleaned_output_path}")